In [27]:
from importlib import reload

In [12]:
# Assuming %autoreload 2 is already active
from src import data_broker
from src import calendar_iterator

In [36]:
import pandas as pd

In [6]:
# 1. Get Data from Module A
tickers = ["AAPL", "MSFT", "NVDA", "AMZN"]
broker = data_broker.DataBroker(tickers=tickers, start_date="2024-01-01", end_date="2025-12-31")
universe_data = broker.fetch_universe_data()

[*********************100%***********************]  4 of 4 completed


In [11]:
# 2. Initialize Module B
calendar = calendar_iterator.CalendarIterator(universe_data, interval="ME") # ME = Month End
rebalance_dates = calendar.generate_rebalance_dates()

print(f"Generated {len(rebalance_dates)} rebalance periods.")
print("First 3 rebalance dates:", [d.strftime('%Y-%m-%d') for d in rebalance_dates[:3]])

Generated 24 rebalance periods.
First 3 rebalance dates: ['2024-01-31', '2024-02-29', '2024-03-28']


## structure varification (general test)

In [13]:
# Pick a random rebalance date to test the snapshot structural footprint
test_date = rebalance_dates[5] # 6th month end
snapshot = calendar.get_historical_snapshot(test_date)

print(f"--- Structural Check for Date: {test_date.strftime('%Y-%m-%d')} ---")
print("Price snapshot shape:", snapshot["price"].shape)
print("Volume snapshot shape:", snapshot["volume"].shape)

# Structural Assertion: The snapshot must end exactly on or right before the test_date
assert snapshot["price"].index[-1] <= test_date
print(" Structural Check Passed: No future data leaked into snapshot.")

--- Structural Check for Date: 2024-06-28 ---
Price snapshot shape: (124, 4)
Volume snapshot shape: (124, 4)
 Structural Check Passed: No future data leaked into snapshot.


In [19]:
test_date

Timestamp('2024-06-28 00:00:00')

In [21]:
snapshot['price'].tail(3)

,AAPL,MSFT,NVDA,AMZN
Date,,,,
2024-06-26,211.418594,445.163940,126.192001,193.610001
2024-06-27,212.261307,445.843231,123.785980,197.850006
2024-06-28,208.811172,440.034515,123.336716,193.250000


## Visually inspect the end of the snapshot matrix to ensure alignment

In [14]:
print("\nFinal rows of the point-in-time snapshot:")
display(snapshot["price"].tail(3))


Final rows of the point-in-time snapshot:


,AAPL,MSFT,NVDA,AMZN
Date,,,,
2024-06-26,211.418594,445.163940,126.192001,193.610001
2024-06-27,212.261307,445.843231,123.785980,197.850006
2024-06-28,208.811172,440.034515,123.336716,193.250000


# test pipeline

In [50]:
reload(strategy_selector_1)

<module 'src.strategy_selector_1' from '/Users/zhigangyu/Documents/Project Z/src/strategy_selector_1.py'>

In [39]:
from src import strategy_selector_1

In [51]:
# Initialize our strategy module (Look back 60 trading days for shorter-term test momentum)
strategy_1 = strategy_selector_1.RiskAdjustedMomentum(lookback_days=60) # Let's find top 25% for a 4-stock universe
strategy_2 = strategy_selector_1.InformationDiscreteMomentum(lookback_days=60) # Let's find top 25% for a 4-stock universe

In [53]:
# Get a point-in-time snapshot from our existing calendar iterator loop
test_date = rebalance_dates[6] # Pick a date mid-way
snapshot = calendar.get_historical_snapshot(test_date)

In [55]:
# Execute the Hot-Swappable strategy
scores_1 = strategy_1.calculate_scores(snapshot)
scores_2 = strategy_2.calculate_scores(snapshot)

In [56]:
scores_1

AAPL    1.00
MSFT    0.50
NVDA    0.75
AMZN    0.25
dtype: float64

In [57]:
scores_2

AAPL    1.00
MSFT    0.50
NVDA    0.75
AMZN    0.25
dtype: float64

In [59]:
# Pull out the exact values manually to check the meaning of the data
prices = snapshot["price"]
curr_p = prices.iloc[-1]
hist_p = prices.iloc[-60]
raw_returns = (curr_p / hist_p) - 1

# Create an investigation DataFrame for visual debugging
diagnostic_df = pd.DataFrame({
    "Price_60_Days_Ago": hist_p,
    "Current_Price": curr_p,
    "Calculated_Return": raw_returns,
    "Strategy1_Output_Score": scores_1,
    "Strategy2_Output_Score": scores_2
}).sort_values(by="Calculated_Return", ascending=False)

print(f"=== Diagnosis Report for Rebalance Date: {test_date.strftime('%Y-%m-%d')} ===")
display(diagnostic_df)

=== Diagnosis Report for Rebalance Date: 2024-07-31 ===


,Price_60_Days_Ago,Current_Price,Calculated_Return,Strategy1_Output_Score,Strategy2_Output_Score
NVDA,91.980827,116.827431,0.270128,0.75,0.75
AAPL,179.905441,220.172760,0.223825,1.00,1.00
MSFT,406.408417,411.877045,0.013456,0.50,0.50
AMZN,188.699997,186.979996,-0.009115,0.25,0.25


# module d

In [67]:
from src import filter_optimizer_1

In [81]:
# Initialize the filter to extract top 25% of our 4-stock universe, 
# demanding at least $5,000,000 in average daily dollar volume.
portfolio_filter = filter_optimizer_1.VolumeWeightFilter(
    top_percent=0.25, 
    volume_window=20, 
    min_dollar_volume=5_000_0, 
    allocation_type="score_weighted"
)

In [82]:
# Ingest continuous scores from our previous Module C execution step
target_weights = portfolio_filter.generate_weights(scores, snapshot)

In [83]:
target_weights

AAPL    1.0
MSFT    0.0
NVDA    0.0
AMZN    0.0
dtype: float64

In [84]:
# check

In [85]:
# Let's inspect the final decision matrix visually
allocation_df = pd.DataFrame({
    "Regression_Score_Rank": scores,
    "Target_Portfolio_Weight": target_weights
}).sort_values(by="Regression_Score_Rank", ascending=False)

print(f"=== Portfolio Weights Vector for {test_date.strftime('%Y-%m-%d')} ===")
display(allocation_df)
print(f"Total Portfolio Exposure: {target_weights.sum() * 100:.2f}%")

=== Portfolio Weights Vector for 2024-07-31 ===


,Regression_Score_Rank,Target_Portfolio_Weight
AAPL,1.00,1.0
NVDA,0.75,0.0
MSFT,0.50,0.0
AMZN,0.25,0.0


Total Portfolio Exposure: 100.00%


# module E: evaluate